In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if __name__ == "__main__":
    model_checkpoint = "cahya/bert2bert-indonesian-summarization"

    # Load sekali aja di luar loop, biar nggak re-download/re-load tiap iterasi
    print("Memuat model, mohon tunggu...")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
    print("Model siap digunakan!\n")

    def kata_ke_token(jumlah_kata):
        return int(jumlah_kata * 1.5)

    def inference_model(input_text, min_length_user, max_length_user):
        input_tokens = tokenizer(input_text, return_tensors="pt", padding="max_length", truncation=True, max_length=512)

        print(f"\n{'-'*100}")
        print("Bentuk Data (Shape):", input_tokens["input_ids"].shape)
        print("5 Token Pertama (Angka):", input_tokens["input_ids"][0][:5])
        print(f"{'-'*100}\n")

        summary_ids = model.generate(
            input_tokens["input_ids"],
            max_length=kata_ke_token(max_length_user),
            min_length=kata_ke_token(min_length_user),
            num_beams=5,              # <- FIXED: tadinya "num_beam"
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        manual_summarize = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        print("--- Hasil Ringkasan (Manual dengan Beam Search) ---")
        print(manual_summarize)

    while True:
        os.system('cls' if os.name == 'nt' else 'clear')
        print(f"\n{'-'*100}")
        print(f"{'Silahkan Ringkas Paragraf Kamu Disini':^100}")
        print(f"{'-'*100}\n")

        try:
            input_text_user = input('Masukkan Paragraf : ')
            min_length_user = int(input('Masukkan minimal kata dalam ringkasan : '))
            max_length_user = int(input('Masukkan maximal kata dalam ringkasan : '))

            daftar_kata = input_text_user.split()
            jumlah_kata = len(daftar_kata)

            if input_text_user == '' or jumlah_kata < 20:
                print("Silahkan ulangi program karena paragraf kamu kurang dari 20 kata")
            elif min_length_user > max_length_user:
                print("Minimal kata tidak boleh lebih besar dari maksimal kata!")
            elif min_length_user <= 0 or max_length_user <= 0:
                print("Minimal/maksimal kata harus lebih besar dari 0!")
            else:
                inference_model(input_text_user, min_length_user, max_length_user)

        except ValueError:
            print("Input minimal/maksimal kata harus berupa angka")

        exit_option = input('\nApakah kamu ingin melanjutkan (y/n) : ').lower()
        if exit_option == 'n':
            break